# Customer Segmentation & Churn Analysis

## Step 2: Data Cleaning

Objectives:
- Remove missing values
- Remove duplicate records
- Remove invalid transactions
- Create new features
- Prepare dataset for customer analysis

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_excel("../data/Online Retail.xlsx")

In [3]:
df.shape

(541909, 8)

In [4]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": round(df.isnull().mean()*100,2)
})

missing

,Missing Values,Percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


## Observation

- `CustomerID` has **135,080 missing values (24.93%)**, so these records will need to be handled before performing customer-level analysis.
- `Description` has **1,454 missing values (0.27%)**, which is a relatively small portion of the dataset.
- All other columns have **no missing values**.

In [8]:
df_clean = df.dropna(subset=["CustomerID"])

In [9]:
df_clean.shape

(406829, 8)

In [10]:
df_clean.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

## Observation

Rows without CustomerID were removed because customer-level analysis is impossible without customer identifiers.

In [11]:
duplicates = df_clean.duplicated().sum()

duplicates

np.int64(5225)

In [12]:
df_clean = df_clean.drop_duplicates()

In [13]:
df_clean.duplicated().sum()

np.int64(0)

## Observation

Duplicate transactions were removed to avoid double counting purchases.

In [14]:
(df_clean["Quantity"] <= 0).sum()

np.int64(8872)

In [15]:
df_clean = df_clean[df_clean["Quantity"] > 0]

In [16]:
(df_clean["Quantity"] <= 0).sum()

np.int64(0)

## Observation

Transactions with zero or negative quantities were removed because they represent returns, cancellations, or invalid entries.

In [17]:
(df_clean["UnitPrice"] <= 0).sum()

np.int64(40)

In [18]:
df_clean = df_clean[df_clean["UnitPrice"] > 0]

In [19]:
(df_clean["UnitPrice"] <= 0).sum()

np.int64(0)

## Observation

Transactions with zero or negative prices were removed because they are not valid sales transactions.

In [20]:
df_clean["TotalPrice"] = df_clean["Quantity"] * df_clean["UnitPrice"]

In [21]:
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## Observation

A new feature called TotalPrice was created to calculate transaction value.

In [22]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

In [23]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 392692 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  object        
 1   StockCode    392692 non-null  object        
 2   Description  392692 non-null  object        
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[us]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  float64       
 7   Country      392692 non-null  str           
 8   TotalPrice   392692 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), object(3), str(1)
memory usage: 30.0+ MB


## Observation

InvoiceDate is stored as datetime, enabling time-based analysis.

In [24]:
df_clean.shape

(392692, 9)

In [25]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 392692 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  object        
 1   StockCode    392692 non-null  object        
 2   Description  392692 non-null  object        
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[us]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  float64       
 7   Country      392692 non-null  str           
 8   TotalPrice   392692 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), object(3), str(1)
memory usage: 30.0+ MB


In [26]:
df_clean.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID,TotalPrice
count,392692.000000,392692,392692.000000,392692.000000,392692.000000
mean,13.119702,2011-07-10 19:13:07.771892,3.125914,15287.843865,22.631500
min,1.000000,2010-12-01 08:26:00,0.001000,12346.000000,0.001000
25%,2.000000,2011-04-07 11:12:00,1.250000,13955.000000,4.950000
50%,6.000000,2011-07-31 12:02:00,1.950000,15150.000000,12.450000
75%,12.000000,2011-10-20 12:53:00,3.750000,16791.000000,19.800000
max,80995.000000,2011-12-09 12:50:00,8142.750000,18287.000000,168469.600000
std,180.492832,NaN,22.241836,1713.539549,311.099224


In [27]:
df_clean.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

## Observation

The cleaned dataset contains only valid customer transactions with no missing values in essential columns.

In [28]:
rows_removed = df.shape[0] - df_clean.shape[0]

print("Original Shape :", df.shape)
print("Cleaned Shape  :", df_clean.shape)
print("Rows Removed   :", rows_removed)

Original Shape : (541909, 8)
Cleaned Shape  : (392692, 9)
Rows Removed   : 149217


## Final Observation

- Original dataset contained 541,909 records.
- After cleaning, 392,692 valid customer transactions remain.
- A total of 149,217 rows were removed due to missing CustomerID, duplicate records, cancelled orders, and invalid prices.
- The cleaned dataset is now ready for exploratory data analysis and feature engineering.

In [29]:
df_clean.to_csv("../data/online_retail_clean.csv", index=False)

In [30]:
import os

os.listdir("../data")

['online_retail_clean.csv',
 'Online Retail.xlsx',
 'customer_rfm_analysis.csv',
 '.ipynb_checkpoints']